# 29: Fine-Tune a Pre-Trained Model

## Day 2: "Can you try one of those transformer things?"

Your TF-IDF baseline from yesterday hit ~95% F1. Your team lead is impressed but asks:
*"I've heard transformers are better at understanding text. Can you try fine-tuning one?"*

Fine-tuning = take a model pre-trained on billions of words and adapt it to YOUR data.
It's like hiring a linguist and teaching them to spot spam — much faster than training from scratch.

## What You'll Learn
- [ ] Tokenize text data for a transformer model
- [ ] Fine-tune DistilBERT for binary text classification
- [ ] Compare frozen vs unfrozen encoder strategies
- [ ] Save and load a trained model for reuse

## Connection to Previous Lessons

| What you learned | How it connects here |
|-----------------|---------------------|
| **Lesson 12**: Training loops (forward → loss → backward → update) | Same loop, but now with a pre-trained model |
| **Lesson 27**: HuggingFace pipelines and tokenizers | Now we go deeper: custom training, not just inference |
| **Lesson 28**: Data cleaning, TF-IDF baseline | We use the same clean data and try to beat TF-IDF |
| **Lesson 30**: Fine-tuning concepts | Now we actually DO it on real data |

In [ ]:
import torch
import torch.nn as nn
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, classification_report
import warnings
warnings.filterwarnings('ignore')
import re

# Check device
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

# CPU-friendly defaults: on a laptop, DistilBERT fine-tuning is slow, so
# we auto-scale the training budget so the notebook is runnable anywhere.
IS_CUDA = device.type == "cuda"
n_epochs = 3 if IS_CUDA else 1
batch_size = 32 if IS_CUDA else 16

print(f"n_epochs={n_epochs}, batch_size={batch_size}  (IS_CUDA={IS_CUDA})")

# Load and prepare data (same cleaning as Notebook 28)
df = pd.read_csv('../../data/sms_spam.csv')
df = df.drop_duplicates(subset='message', keep='first').copy()
df['is_spam'] = (df['label'] == 'spam').astype(int)

# Split
train_texts, test_texts, train_labels, test_labels = train_test_split(
    df['message'].tolist(),
    df['is_spam'].tolist(),
    test_size=0.2,
    random_state=42,
    stratify=df['is_spam']
)

# Further split train into train + validation
train_texts, val_texts, train_labels, val_labels = train_test_split(
    train_texts, train_labels,
    test_size=0.15,
    random_state=42,
    stratify=train_labels
)

print(f"Train: {len(train_texts)}, Val: {len(val_texts)}, Test: {len(test_texts)}")
print(f"Spam ratio — Train: {np.mean(train_labels):.1%}, Val: {np.mean(val_labels):.1%}, Test: {np.mean(test_labels):.1%}")
print("✓ Data loaded and split")

## 1. Tokenization: Turning Text into Numbers

Transformers don't read text — they read **token IDs**. A tokenizer converts text into the specific format the model expects: token IDs, attention masks, and special tokens.

In [ ]:
from transformers import DistilBertTokenizer

tokenizer = DistilBertTokenizer.from_pretrained('distilbert-base-uncased')

# See how tokenization works
example = "Free entry to WIN a £1000 prize! Text YES to claim now!"
tokens = tokenizer(example, return_tensors='pt', padding=True, truncation=True, max_length=128)

print(f"Original:   '{example}'")
print(f"Token IDs:  {tokens['input_ids'][0][:20].tolist()}...")
print(f"Tokens:     {tokenizer.convert_ids_to_tokens(tokens['input_ids'][0][:20])}")
print(f"Attention:  {tokens['attention_mask'][0][:20].tolist()}...")
print()
print(f"Total tokens: {tokens['input_ids'].shape[1]}")
print(f"Vocab size: {tokenizer.vocab_size:,}")
print()
print("Notice:")
print("   [CLS] = start token, [SEP] = end token")
print("   '£1000' gets split into subwords: '£', '1000'")
print("   Unknown words get split into pieces the model knows")

In [ ]:
# --- PyTorch Dataset for SMS messages ---
class SMSDataset(Dataset):
    """Wraps SMS texts + labels for PyTorch DataLoader."""
    
    def __init__(self, texts, labels, tokenizer, max_length=128):
        self.texts = texts
        self.labels = labels
        self.tokenizer = tokenizer
        self.max_length = max_length
    
    def __len__(self):
        return len(self.texts)
    
    def __getitem__(self, idx):
        encoding = self.tokenizer(
            self.texts[idx],
            padding='max_length',
            truncation=True,
            max_length=self.max_length,
            return_tensors='pt'
        )
        return {
            'input_ids': encoding['input_ids'].squeeze(),
            'attention_mask': encoding['attention_mask'].squeeze(),
            'label': torch.tensor(self.labels[idx], dtype=torch.long)
        }

# Create datasets and dataloaders
train_dataset = SMSDataset(train_texts, train_labels, tokenizer)
val_dataset = SMSDataset(val_texts, val_labels, tokenizer)
test_dataset = SMSDataset(test_texts, test_labels, tokenizer)

train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=batch_size)
test_loader = DataLoader(test_dataset, batch_size=batch_size)

# Verify
batch = next(iter(train_loader))
print(f"Batch shapes:")
print(f"  input_ids:      {batch['input_ids'].shape}")
print(f"  attention_mask:  {batch['attention_mask'].shape}")
print(f"  labels:          {batch['label'].shape}")
print(f"\n✓ DataLoaders ready")

## 2. The Model: DistilBERT + Classification Head

DistilBERT is a smaller, faster version of BERT (66M vs 110M parameters).
We add a simple classification head on top: `DistilBERT → Linear → 2 classes`.

**Strategy 1: Frozen encoder** — only train the classification head (fast, less risk)
**Strategy 2: Full fine-tune** — train everything (slower, potentially better)

In [ ]:
from transformers import DistilBertModel

class SpamClassifier(nn.Module):
    """DistilBERT encoder + classification head for spam detection."""
    
    def __init__(self, freeze_encoder=True):
        super().__init__()
        self.encoder = DistilBertModel.from_pretrained('distilbert-base-uncased')
        self.classifier = nn.Sequential(
            nn.Linear(768, 128),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(128, 2)
        )
        
        # Optionally freeze the encoder
        if freeze_encoder:
            for param in self.encoder.parameters():
                param.requires_grad = False
    
    def forward(self, input_ids, attention_mask):
        # Get encoder output
        output = self.encoder(input_ids=input_ids, attention_mask=attention_mask)
        # Use [CLS] token representation (first token)
        cls_output = output.last_hidden_state[:, 0, :]
        # Classify
        logits = self.classifier(cls_output)
        return logits

# Create frozen model first (Strategy 1)
model_frozen = SpamClassifier(freeze_encoder=True).to(device)

total_params = sum(p.numel() for p in model_frozen.parameters())
trainable_params = sum(p.numel() for p in model_frozen.parameters() if p.requires_grad)
print(f"Total parameters:     {total_params:>12,}")
print(f"Trainable parameters: {trainable_params:>12,}")
print(f"Frozen parameters:    {total_params - trainable_params:>12,}")
print(f"\nOnly {trainable_params/total_params:.1%} of parameters are trainable!")
print(f"The rest are pre-trained knowledge we're reusing.")

In [ ]:
def train_epoch(model, loader, optimizer, criterion, device):
    """Train for one epoch. Returns average loss."""
    model.train()
    total_loss = 0
    
    for batch in loader:
        input_ids = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        labels = batch['label'].to(device)
        
        # Forward pass
        outputs = model(input_ids, attention_mask)
        loss = criterion(outputs, labels)
        
        # Backward pass
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        
        total_loss += loss.item()
    
    return total_loss / len(loader)

def evaluate(model, loader, device):
    """Evaluate model. Returns predictions and true labels."""
    model.eval()
    all_preds = []
    all_labels = []
    
    with torch.no_grad():
        for batch in loader:
            input_ids = batch['input_ids'].to(device)
            attention_mask = batch['attention_mask'].to(device)
            labels = batch['label'].to(device)
            
            outputs = model(input_ids, attention_mask)
            preds = outputs.argmax(dim=1)
            
            all_preds.extend(preds.cpu().tolist())
            all_labels.extend(labels.cpu().tolist())
    
    return all_preds, all_labels

print("✓ Training and evaluation functions defined")

## 3. Strategy 1: Frozen Encoder (Fast & Safe)

Train only the classification head. The DistilBERT encoder stays exactly as pre-trained.
This is fast (~1 min) and usually gives decent results.

In [ ]:
# --- Strategy 1: Frozen Encoder ---
optimizer = torch.optim.Adam(model_frozen.parameters(), lr=1e-3)
criterion = nn.CrossEntropyLoss()

frozen_epochs = 5 if IS_CUDA else 2
print("Training with FROZEN encoder (only classification head learns)...")
print("-" * 50)

frozen_history = {'train_loss': [], 'val_f1': []}

for epoch in range(frozen_epochs):
    train_loss = train_epoch(model_frozen, train_loader, optimizer, criterion, device)
    val_preds, val_labels_out = evaluate(model_frozen, val_loader, device)
    val_f1 = f1_score(val_labels_out, val_preds)
    
    frozen_history['train_loss'].append(train_loss)
    frozen_history['val_f1'].append(val_f1)
    
    print(f"  Epoch {epoch+1}/{frozen_epochs} — loss: {train_loss:.4f}, val F1: {val_f1:.3f}")

# Test evaluation
test_preds, test_labels_out = evaluate(model_frozen, test_loader, device)
frozen_f1 = f1_score(test_labels_out, test_preds)
frozen_acc = accuracy_score(test_labels_out, test_preds)
print(f"\nFrozen model test results:")
print(f"  Accuracy: {frozen_acc:.3f}")
print(f"  F1 Score: {frozen_f1:.3f}")
print(f"\n✓ Strategy 1 complete")

## What Can Go Wrong: Catastrophic Forgetting

What if we use a learning rate that's TOO HIGH for fine-tuning?

The pre-trained weights get overwritten with noise — the model "forgets" English!
This is called **catastrophic forgetting**.

In [ ]:
# --- Catastrophic Forgetting Demo ---
bad_model = SpamClassifier(freeze_encoder=False).to(device)
bad_optimizer = torch.optim.Adam(bad_model.parameters(), lr=0.1)  # WAY too high!

print("Training with lr=0.1 (TOO HIGH)...")
print("-" * 50)

for epoch in range(3):
    train_loss = train_epoch(bad_model, train_loader, bad_optimizer, criterion, device)
    val_preds_bad, val_labels_bad = evaluate(bad_model, val_loader, device)
    val_acc_bad = accuracy_score(val_labels_bad, val_preds_bad)
    print(f"  Epoch {epoch+1}/3 — loss: {train_loss:.4f}, val accuracy: {val_acc_bad:.3f}")

print()
print("Accuracy ~50% = random guessing!")
print("The model 'forgot' everything it learned from billions of words.")
print()
print("Rule of thumb for fine-tuning learning rates:")
print("   Classifier head: 1e-3 to 1e-4")
print("   Full model:      2e-5 to 5e-5  (100x smaller!)")

del bad_model  # Free memory

## 4. Strategy 2: Full Fine-Tune (with proper learning rate)

Now unfreeze the encoder and train EVERYTHING — but with a much smaller learning rate.
The pre-trained weights need gentle updates, not aggressive rewrites.

In [ ]:
# --- Strategy 2: Full Fine-Tune ---
model_full = SpamClassifier(freeze_encoder=False).to(device)

# Different learning rates for encoder vs classifier
optimizer_full = torch.optim.AdamW([
    {'params': model_full.encoder.parameters(), 'lr': 2e-5},
    {'params': model_full.classifier.parameters(), 'lr': 1e-3},
], weight_decay=0.01)

print("Training with FULL fine-tune (different LR per layer)...")
print("-" * 50)

full_history = {'train_loss': [], 'val_f1': []}
best_val_f1 = 0
best_state = None

for epoch in range(n_epochs):
    train_loss = train_epoch(model_full, train_loader, optimizer_full, criterion, device)
    val_preds_f, val_labels_f = evaluate(model_full, val_loader, device)
    val_f1 = f1_score(val_labels_f, val_preds_f)
    
    full_history['train_loss'].append(train_loss)
    full_history['val_f1'].append(val_f1)
    
    # Save best model
    if val_f1 > best_val_f1:
        best_val_f1 = val_f1
        best_state = {k: v.clone() for k, v in model_full.state_dict().items()}
        print(f"  Epoch {epoch+1}/{n_epochs} — loss: {train_loss:.4f}, val F1: {val_f1:.3f} <- best!")
    else:
        print(f"  Epoch {epoch+1}/{n_epochs} — loss: {train_loss:.4f}, val F1: {val_f1:.3f}")

# Load best model
model_full.load_state_dict(best_state)

# Test
test_preds_full, test_labels_full = evaluate(model_full, test_loader, device)
full_f1 = f1_score(test_labels_full, test_preds_full)
full_acc = accuracy_score(test_labels_full, test_preds_full)
print(f"\nFull fine-tune test results:")
print(f"  Accuracy: {full_acc:.3f}")
print(f"  F1 Score: {full_f1:.3f}")
print(f"\n✓ Strategy 2 complete")

In [ ]:
# --- Comparison ---
print("=" * 50)
print("Strategy Comparison")
print("=" * 50)
print(f"{'Strategy':<25} {'Accuracy':>10} {'F1':>10}")
print("-" * 50)
print(f"{'Frozen encoder':<25} {frozen_acc:>10.3f} {frozen_f1:>10.3f}")
print(f"{'Full fine-tune':<25} {full_acc:>10.3f} {full_f1:>10.3f}")
print("-" * 50)
print(f"{'TF-IDF baseline (L28)':<25} {'~0.970':>10} {'~0.950':>10}")
print()

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Loss curves
axes[0].plot(frozen_history['train_loss'], 'b-o', label='Frozen')
axes[0].plot(full_history['train_loss'], 'r-o', label='Full fine-tune')
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Training Loss')
axes[0].set_title('Training Loss')
axes[0].legend()

# F1 curves
axes[1].plot(frozen_history['val_f1'], 'b-o', label='Frozen')
axes[1].plot(full_history['val_f1'], 'r-o', label='Full fine-tune')
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('Validation F1')
axes[1].set_title('Validation F1')
axes[1].legend()

plt.tight_layout()
plt.show()

print("Full fine-tune usually wins, but takes longer and risks overfitting")
print("On small datasets, frozen encoder can be surprisingly competitive")

## 5. Save and Load: Making It Reusable

A model that only exists in a running notebook is useless.
Save it so you (or your teammate) can load it tomorrow.

In [ ]:
import os

# Save the best model
save_dir = '../../data/spam_model'
os.makedirs(save_dir, exist_ok=True)

# Save model weights
torch.save(model_full.state_dict(), os.path.join(save_dir, 'model.pt'))

# Save tokenizer (so we always use the matching one)
tokenizer.save_pretrained(save_dir)

print(f"✓ Model saved to {save_dir}/")
print(f"  model.pt: {os.path.getsize(os.path.join(save_dir, 'model.pt')) / 1e6:.1f} MB")

# --- Load it back (simulate a fresh start) ---
loaded_model = SpamClassifier(freeze_encoder=False)
loaded_model.load_state_dict(torch.load(os.path.join(save_dir, 'model.pt'), map_location=device, weights_only=True))
loaded_model = loaded_model.to(device)
loaded_model.eval()

# Verify it works
loaded_tokenizer = DistilBertTokenizer.from_pretrained(save_dir)

def predict_message(text, model=loaded_model, tok=loaded_tokenizer):
    """Classify a single message. Returns (label, confidence)."""
    encoding = tok(text, return_tensors='pt', padding=True, truncation=True, max_length=128)
    encoding = {k: v.to(device) for k, v in encoding.items()}
    
    with torch.no_grad():
        logits = model(encoding['input_ids'], encoding['attention_mask'])
        probs = torch.softmax(logits, dim=1)
        pred = probs.argmax(dim=1).item()
        conf = probs[0, pred].item()
    
    return ('spam' if pred == 1 else 'ham'), conf

# Test predictions
test_msgs = [
    "Hey, are you free for dinner tonight?",
    "CONGRATULATIONS! You've won a FREE iPhone! Click here NOW!",
    "Meeting moved to 3pm tomorrow. Can you make it?",
    "Txt STOP to cancel. Ur subscribed to premium service £3/week",
]

print("\nLoaded model predictions:")
print("-" * 60)
for msg in test_msgs:
    label, conf = predict_message(msg)
    indicator = "SPAM" if label == 'spam' else " ok "
    print(f"  [{indicator} {conf:.0%}] {msg[:55]}...")
print("\n✓ Model loaded and working!")

## What Did It Learn? Attention Visualization

After fine-tuning, we can peek at the attention weights of the last encoder layer to see *which* tokens the model focuses on when it decides "spam" vs "ham".

Each cell in the heatmap below shows how strongly token in the row "attends to" the token in the column. Brighter = more attention.
Watch for spam examples focusing on money/urgency words.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

# 4 sample messages: 2 spam + 2 ham
df_sample = pd.DataFrame({
    'text': [
        'Free entry in 2 weekly competition win £1000',
        'WINNER!! You have won a $1000 Walmart gift card. Click to claim',
        'Hey are we still meeting for lunch tomorrow?',
        'Ok cool thanks, see you at 6',
    ],
    'label': ['spam', 'spam', 'ham', 'ham'],
})

# Use the fine-tuned model (model_full) with output_attentions=True.
# SpamClassifier wraps DistilBertModel — we call the encoder directly
# to get attention weights; the classifier head doesn't produce them.
model_full.eval()

fig, axes = plt.subplots(2, 2, figsize=(14, 10))

for ax, (_, row) in zip(axes.flat, df_sample.iterrows()):
    inputs = tokenizer(
        row['text'],
        return_tensors='pt',
        truncation=True,
        max_length=32,
        padding=True,
    ).to(device)

    with torch.no_grad():
        enc_out = model_full.encoder(
            input_ids=inputs['input_ids'],
            attention_mask=inputs['attention_mask'],
            output_attentions=True,
        )

    # enc_out.attentions: tuple of (n_layers,) each [batch, n_heads, seq, seq]
    # Take last layer, average over heads
    attn = enc_out.attentions[-1][0].mean(dim=0).cpu().numpy()  # (seq_len, seq_len)
    tokens = tokenizer.convert_ids_to_tokens(inputs['input_ids'][0])

    # Trim to actual (non-padding) length
    seq_len = inputs['attention_mask'][0].sum().item()
    attn = attn[:seq_len, :seq_len]
    tokens = tokens[:seq_len]

    im = ax.imshow(attn, cmap='Blues', aspect='auto')
    ax.set_xticks(range(len(tokens)))
    ax.set_xticklabels(tokens, rotation=45, ha='right', fontsize=8)
    ax.set_yticks(range(len(tokens)))
    ax.set_yticklabels(tokens, fontsize=8)
    ax.set_title(f"{row['label'].upper()}: {row['text'][:45]}", fontsize=9)
    plt.colorbar(im, ax=ax, fraction=0.046, pad=0.04)

plt.suptitle(
    "Last-layer attention (averaged over heads) — what the fine-tuned model looks at",
    fontsize=11
)
plt.tight_layout()
plt.show()

print("Notice: spam examples often concentrate attention on money/prize words (£, won, claim).")
print("Ham examples tend to spread attention more uniformly across tokens.")

## 📝 Exercises

In [ ]:
# =================================================================
# Exercise 1: Learning Rate Sweep (frozen encoder — fast on CPU!)
# =================================================================
#
# Full fine-tuning takes minutes per run on CPU. But if we freeze
# the encoder and only train the tiny classification head, each run
# is 10-50x faster — the lesson about learning rates is the same.
#
# Your task: loop over 3 learning rates, train a frozen-head model
# for a fixed number of steps, evaluate accuracy, and pick the best.
#
# Steps:
#   1. For each lr in lrs_to_try:
#      - Create a new SpamClassifier(freeze_encoder=True).to(device)
#      - Create AdamW optimizer on trainable params only
#      - Run the training loop below for TRAIN_STEPS steps
#      - Evaluate accuracy on test_loader
#      - Store in results[lr]
#   2. Set best_frozen_acc and best_frozen_lr from results
#
# Hint: the model's forward() returns logits (shape [B, 2])
#       compute loss with: criterion(logits, labels)
#       labels come from batch['label'], not batch['labels']
# =================================================================

TRAIN_STEPS = 150  # ~10-30 sec on CPU per LR

lrs_to_try = [1e-5, 5e-4, 5e-2]
results = {}
best_frozen_acc = None
best_frozen_lr = None

# YOUR CODE HERE:
for lr in lrs_to_try:
    # 1. Create frozen model
    # m = SpamClassifier(freeze_encoder=True).to(device)
    # 2. Optimizer on trainable params only
    # opt = torch.optim.AdamW([p for p in m.parameters() if p.requires_grad], lr=lr)
    # 3. Training loop
    # m.train()
    # for step, batch in enumerate(train_loader):
    #     if step >= TRAIN_STEPS: break
    #     input_ids = batch['input_ids'].to(device)
    #     attn     = batch['attention_mask'].to(device)
    #     labels   = batch['label'].to(device)
    #     logits   = m(input_ids, attn)
    #     loss     = criterion(logits, labels)
    #     loss.backward(); opt.step(); opt.zero_grad()
    # 4. Evaluate
    # preds, true = evaluate(m, test_loader, device)
    # results[lr] = accuracy_score(true, preds)
    pass

# After your loop, pick the best:
# best_frozen_lr  = max(results, key=results.get)
# best_frozen_acc = results[best_frozen_lr]

# =================================================================
# TESTS
# =================================================================
assert best_frozen_acc is not None and best_frozen_lr is not None, \
    "Fill in your code and set best_frozen_acc / best_frozen_lr!"
assert len(results) == 3, f"Expected results for 3 LRs, got {len(results)}"
assert best_frozen_acc > 0.80, (
    f"Expected best accuracy > 0.80, got {best_frozen_acc:.3f}. "
    f"Results per LR: { {f'{lr:.0e}': f'{acc:.3f}' for lr, acc in results.items()} }"
)

print("Results:")
for lr in lrs_to_try:
    bar = "█" * int(results[lr] * 30)
    note = " ← best!" if lr == best_frozen_lr else ""
    print(f"  LR={lr:.0e}: acc={results[lr]:.3f} {bar}{note}")
print(f"\nBest LR: {best_frozen_lr:.0e} → accuracy {best_frozen_acc:.3f}")
print("Observation: middle LR (5e-4) usually wins — 1e-5 is too slow, 5e-2 diverges.")
print("\n✓ Exercise 1 complete!")

In [ ]:
# =================================================================
# Exercise 2: Test the Model on Your Own Messages
# =================================================================
#
# Write 5 messages: 3 ham and 2 spam.
# Run them through the model and check if it gets them right.
# Try to FOOL the model — can you write a spam it misclassifies?
#
# Hints:
#   - Use predict_message(text) from above
#   - Try edge cases: legitimate messages with "free" or "call"
#   - Try subtle spam: no obvious keywords but still suspicious
# =================================================================

# YOUR CODE HERE:
my_messages = [
    # (message, expected_label)
    # ("your message here", "ham"),   # or "spam"
]

# =================================================================
# TESTS
# =================================================================
assert len(my_messages) >= 5, f"Write at least 5 messages, got {len(my_messages)}"
assert all(label in ('ham', 'spam') for _, label in my_messages), "Labels must be 'ham' or 'spam'"
assert sum(1 for _, l in my_messages if l == 'ham') >= 2, "Include at least 2 ham messages"
assert sum(1 for _, l in my_messages if l == 'spam') >= 2, "Include at least 2 spam messages"
print("✓ Messages validated\n")

correct = 0
print(f"{'Pred':>6} {'Conf':>6} {'Expected':>8}  {'Match':>5}  Message")
print("-" * 70)
for msg, expected in my_messages:
    pred, conf = predict_message(msg)
    match = pred == expected
    correct += match
    icon = "✓" if match else "✗"
    print(f"  {pred:>4} {conf:>5.0%}  {expected:>8}   {icon}    {msg[:42]}...")

print(f"\nAccuracy on your messages: {correct}/{len(my_messages)} ({correct/len(my_messages):.0%})")

if correct == len(my_messages):
    print("Perfect! Now try writing harder examples to find the model's limits.")
else:
    print(f"The model missed {len(my_messages) - correct} — look at confidence scores for clues")

print("\n✓ Exercise 2 complete!")

## Summary: Day 2

You fine-tuned a pre-trained transformer on real data:

| Strategy | What it does | When to use |
|----------|-------------|-------------|
| **Frozen encoder** | Only train the classifier head | Small data, quick experiment, limited compute |
| **Full fine-tune** | Train everything with small LR | More data, need max performance |
| **High LR fine-tune** | Destroys the model | Never! (catastrophic forgetting demo) |

**Key takeaways:**
1. **Tokenization** converts text to token IDs + attention masks
2. **Learning rate** is the most critical hyperparameter — 2e-5 is a safe default
3. **Save your model** — a model only in a notebook is useless
4. **Compare strategies** — frozen encoder is surprisingly good for small datasets

**Next up**: Lesson 30 — How to evaluate properly (beyond accuracy) →